In [31]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
import joblib
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

In [32]:
data = pd.read_csv('data.csv')
data = data.sample(frac=1, random_state=42).reset_index(drop=True)

test_subjects = [11, 12, 13]
train_df = data[~data['subject'].isin(test_subjects)].copy()
val_df = data[data['subject'].isin(test_subjects)].copy()


In [33]:
landmark_names = [
    'nose', 'left_eye_inner', 'left_eye', 'left_eye_outer',
    'right_eye_inner', 'right_eye', 'right_eye_outer',
    'left_ear', 'right_ear',
    'mouth_left', 'mouth_right',
    'left_shoulder', 'right_shoulder',
    'left_elbow', 'right_elbow'
]

upper_cols = []
for name in landmark_names:
    upper_cols.extend([f'{name}_x', f'{name}_y', f'{name}_z'])

In [34]:
def compute_features(pts):
    nose = pts[0]
    left_shoulder = pts[11]
    right_shoulder = pts[12]
    left_elbow = pts[13]
    right_elbow = pts[14]

    shoulder_mid = (left_shoulder + right_shoulder) / 2.0
    shoulder_width = np.linalg.norm(left_shoulder - right_shoulder)
    if shoulder_width < 1e-6:
        shoulder_width = 1.0

    f = []

    vec = nose - shoulder_mid
    vertical = np.array([0, 1, 0])
    cos_angle = np.dot(vec, vertical) / (np.linalg.norm(vec) * np.linalg.norm(vertical) + 1e-8)
    f.append(np.degrees(np.arccos(np.clip(cos_angle, -1.0, 1.0))))

    shoulder_vec = right_shoulder - left_shoulder
    horizontal = np.array([1, 0, 0])
    cos_slope = np.dot(shoulder_vec, horizontal) / (np.linalg.norm(shoulder_vec) * np.linalg.norm(horizontal) + 1e-8)
    f.append(np.degrees(np.arccos(np.clip(cos_slope, -1.0, 1.0))))

    f.append(np.linalg.norm(nose - shoulder_mid) / shoulder_width)

    f.append((left_elbow[2] - left_shoulder[2]) / shoulder_width)
    f.append((left_elbow[0] - left_shoulder[0]) / shoulder_width)
    f.append((left_elbow[1] - left_shoulder[1]) / shoulder_width)
    f.append(np.linalg.norm(left_elbow - left_shoulder) / shoulder_width)

    f.append((right_elbow[2] - right_shoulder[2]) / shoulder_width)
    f.append((right_elbow[0] - right_shoulder[0]) / shoulder_width)
    f.append((right_elbow[1] - right_shoulder[1]) / shoulder_width)
    f.append(np.linalg.norm(right_elbow - right_shoulder) / shoulder_width)

    f.append((left_shoulder[1] - right_shoulder[1]) / shoulder_width)

    vec_proj = vec.copy()
    vec_proj[1] = 0
    shoulder_proj = shoulder_vec.copy()
    shoulder_proj[1] = 0
    if np.linalg.norm(vec_proj) > 0 and np.linalg.norm(shoulder_proj) > 0:
        cos_rot = np.dot(vec_proj, shoulder_proj) / (np.linalg.norm(vec_proj) * np.linalg.norm(shoulder_proj) + 1e-8)
        f.append(np.degrees(np.arccos(np.clip(cos_rot, -1.0, 1.0))))
    else:
        f.append(0.0)

    f.append(np.linalg.norm(left_elbow - right_elbow) / shoulder_width)

    return np.array(f)


def extract_features(df):
    X = df[upper_cols].values.reshape(-1, len(landmark_names), 3)
    return np.array([compute_features(pts) for pts in X])


X_train_raw = extract_features(train_df)
X_val_raw = extract_features(val_df)

y_train = (train_df['upperbody_label'] != 'TLB').astype(int)
y_val = (val_df['upperbody_label'] != 'TLB').astype(int)

In [35]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_val = scaler.transform(X_val_raw)

In [36]:
def augment(X, y, target_ratio=1.0, noise_std=0.03):
    X_good = X[y == 0]
    X_bad = X[y == 1]

    n_good = len(X_good)
    n_bad = len(X_bad)

    target_good = int(n_bad * target_ratio)

    if target_good <= n_good:
        return X, y

    needed = target_good - n_good

    idx = np.random.choice(n_good, needed, replace=True)

    synthetic = (
        X_good[idx]
        + np.random.normal(0, noise_std, (needed, X.shape[1]))
    )

    X_new = np.vstack([X, synthetic])
    y_new = np.hstack([y, np.zeros(needed)])

    return X_new, y_new


X_train_aug, y_train_aug = augment(X_train, y_train)

In [49]:
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix

best_params["scale_pos_weight"] = 5
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500, step=50),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.5, 2.0),
       # 'scale_pos_weight': trial.suggest_float('scale_pos_weight', 0.5, 2.0),
        'random_state': 42,
        'n_jobs': -1,
        'eval_metric': 'logloss',
        'early_stopping_rounds': 20,
    }
    clf = XGBClassifier(**params)
    clf.fit(X_train_aug, y_train_aug,
            eval_set=[(X_val, y_val)],
            verbose=False)
    preds = clf.predict(X_val)
    return f1_score(y_val, preds, average='weighted')

study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=42),
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=10)
)
study.optimize(objective, n_trials=100, timeout=600)

best_params = study.best_params
print("Best parameters:", best_params)
print("Best validation F1:", study.best_value)

best_clf = XGBClassifier(**best_params, random_state=42, n_jobs=-1,
                          eval_metric='logloss', early_stopping_rounds=20)
best_clf.fit(X_train_aug, y_train_aug,
             eval_set=[(X_val, y_val)],
             verbose=False)

probs = best_clf.predict_proba(X_val)[:, 1]
preds = best_clf.predict(X_val)

print("\n── Final Evaluation ──────────────────────────")
print(classification_report(y_val, preds, target_names=['Good', 'Bad']))
print("Confusion matrix:")
print(confusion_matrix(y_val, preds))

print("\n── Threshold Sweep ───────────────────────────")
for t in np.arange(0.2, 0.8, 0.05):
    p = (probs >= t).astype(int)
    print(f"  t={t:.2f}  weighted={f1_score(y_val, p, average='weighted'):.4f}  "
          f"Good={f1_score(y_val, p, average=None)[0]:.4f}  "
          f"Bad={f1_score(y_val, p, average=None)[1]:.4f}")

joblib.dump(best_clf, 'posture_model_optuna.pkl')
joblib.dump(scaler, 'scaler_optuna.pkl')

[I 2026-06-18 15:52:31,686] A new study created in memory with name: no-name-c6a32c83-096d-4a46-bc30-a1a1b875a851
[I 2026-06-18 15:52:32,024] Trial 0 finished with value: 0.9041530244025525 and parameters: {'n_estimators': 200, 'max_depth': 10, 'learning_rate': 0.1205712628744377, 'subsample': 0.7993292420985183, 'colsample_bytree': 0.5780093202212182, 'min_child_weight': 2, 'gamma': 0.2904180608409973, 'reg_alpha': 0.8661761457749352, 'reg_lambda': 1.4016725176148133}. Best is trial 0 with value: 0.9041530244025525.
[I 2026-06-18 15:52:32,312] Trial 1 finished with value: 0.8900808464378034 and parameters: {'n_estimators': 400, 'max_depth': 3, 'learning_rate': 0.2708160864249968, 'subsample': 0.9162213204002109, 'colsample_bytree': 0.6061695553391381, 'min_child_weight': 2, 'gamma': 0.9170225492671691, 'reg_alpha': 0.3042422429595377, 'reg_lambda': 1.2871346474483567}. Best is trial 0 with value: 0.9041530244025525.
[I 2026-06-18 15:52:32,690] Trial 2 finished with value: 0.8862968920

Best parameters: {'n_estimators': 350, 'max_depth': 10, 'learning_rate': 0.06995408838341347, 'subsample': 0.6442172854267176, 'colsample_bytree': 0.5766118018176974, 'min_child_weight': 1, 'gamma': 0.43148235914730837, 'reg_alpha': 0.7076869755643839, 'reg_lambda': 1.8255334210503322}
Best validation F1: 0.9089453731824249

── Final Evaluation ──────────────────────────
              precision    recall  f1-score   support

        Good       0.70      0.82      0.75       143
         Bad       0.96      0.93      0.94       667

    accuracy                           0.91       810
   macro avg       0.83      0.87      0.85       810
weighted avg       0.91      0.91      0.91       810

Confusion matrix:
[[117  26]
 [ 50 617]]

── Threshold Sweep ───────────────────────────
  t=0.20  weighted=0.8859  Good=0.6311  Bad=0.9405
  t=0.25  weighted=0.9132  Good=0.7373  Bad=0.9509
  t=0.30  weighted=0.9237  Good=0.7798  Bad=0.9546
  t=0.35  weighted=0.9129  Good=0.7560  Bad=0.9466
  t=0.

['scaler_optuna.pkl']

In [52]:
best_clf = XGBClassifier(
    **best_params,
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=5,
)

In [53]:
from sklearn.calibration import CalibratedClassifierCV

In [54]:
calibrated_model = CalibratedClassifierCV(
    best_clf,
    method="isotonic",   # or "sigmoid"
    cv=5
)

calibrated_model.fit(X_train, y_train)

CalibratedClassifierCV(cv=5,
                       estimator=XGBClassifier(base_score=None, booster=None,
                                               callbacks=None,
                                               colsample_bylevel=None,
                                               colsample_bynode=None,
                                               colsample_bytree=0.5766118018176974,
                                               device=None,
                                               early_stopping_rounds=None,
                                               enable_categorical=False,
                                               eval_metric=None,
                                               feature_types=None,
                                               feature_weights=None,
                                               gamma=0.43148235914730837,
                                               grow_policy=None,
                                               importance_type=None,
                                               interaction_constraints=None,
                                               learning_rate=0.06995408838341347,
                                               max_bin=None,
                                               max_cat_threshold=None,
                                               max_cat_to_onehot=None,
                                               max_delta_step=None,
                                               max_depth=10, max_leaves=None,
                                               min_child_weight=1, missing=nan,
                                               monotone_constraints=None,
                                               multi_strategy=None,
                                               n_estimators=350, n_jobs=-1,
                                               num_parallel_tree=None, ...),
                       method='isotonic')

In [55]:





joblib.dump(calibrated_model, 'posture_model_optuna.pkl')
joblib.dump(scaler, 'scaler_optuna.pkl')

['scaler_optuna.pkl']